# 03 · Feature engineering and preprocessing
Card-history features use **only earlier transactions of the same card** and **never the fraud label**
(labels arrive weeks later in reality). Implementation: `src/features.py`; preprocessing: `src/preprocessing.py`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
from src import config
from src.features import build_feature_table, model_features
df = build_feature_table()
model_features()

['amt',
 'age',
 'city_pop',
 'is_weekend',
 'card_txn_count_1h',
 'card_txn_count_24h',
 'secs_since_prev_txn',
 'amt_to_card_mean_ratio',
 'amt_zscore_card',
 'card_category_share',
 'card_prior_txn_count',
 'hour',
 'day_of_week',
 'category']

## Feature catalogue
| Feature | Definition | Why it may help |
|---|---|---|
| `amt` | transaction amount | fraud amounts are bimodal: small card tests and large purchases |
| `category` | merchant category | online shopping / misc_net / grocery_pos carry higher fraud rates |
| `hour`, `day_of_week`, `is_weekend` | from timestamp | fraud concentrates late at night |
| `age` | age at transaction | fraud rate varies by age band |
| `city_pop` | cardholder city population | proxy for urban/rural context |
| `card_txn_count_1h`, `card_txn_count_24h` | earlier txns of the card in the window | fraud comes in bursts |
| `secs_since_prev_txn` | time since the card's previous txn | bursts mean short gaps |
| `amt_to_card_mean_ratio` | amount / card's average so far | unusual spend for *this* customer |
| `amt_zscore_card` | (amount − card mean) / card std | same idea, scale-aware |
| `card_category_share` | share of earlier txns in this category | unfamiliar category for this customer |
| `card_prior_txn_count` | number of earlier txns | account tenure (sensitive to a dataset artefact; ablated) |

## Verification against a brute-force recomputation for one card

In [2]:
c = df.cc_num.iloc[0]; d = df[df.cc_num == c].sort_values('ts').reset_index(drop=True); i = 500; t = d.ts[i]
print('24h count  naive:', int(((d.ts >= t - pd.Timedelta('24h')) & (d.index < i)).sum()), '| feature:', d.card_txn_count_24h[i])
print('mean ratio naive: %.4f | feature: %.4f' % (d.amt[i] / d.amt[:i].mean(), d.amt_to_card_mean_ratio[i]))

24h count  naive: 8 | feature: 8
mean ratio naive: 0.5082 | feature: 0.5082


## How well does each feature separate the classes on its own? (train)

In [3]:
from src.inspect_data import rank_auc
tr = df[df.split == 'train']; y = tr[config.TARGET].to_numpy()
pd.DataFrame({f: {'univariate_auc': rank_auc(y, tr[f].fillna(tr[f].median()).to_numpy())} for f in model_features() if f != 'category'}).T.sort_values('univariate_auc')

,univariate_auc
secs_since_prev_txn,0.352607
card_prior_txn_count,0.353816
is_weekend,0.488557
card_category_share,0.493865
day_of_week,0.504908
city_pop,0.505041
age,0.542733
card_txn_count_24h,0.567164
hour,0.585102
card_txn_count_1h,0.645591


## Preprocessing pipelines
Linear model: impute → signed log for skewed columns → standardise; one-hot for category/hour/weekday.
Trees: impute only; ordinal-encode category. Both are fitted inside a `Pipeline`, i.e. on training folds only.

In [4]:
from src.preprocessing import build_preprocessor
for kind in ('linear', 'tree'):
    pre = build_preprocessor(kind, model_features()).fit(tr[model_features()].head(50000))
    print(kind, '->', len(pre.get_feature_names_out()), 'model inputs'); print(list(pre.get_feature_names_out())[:20], '...')

linear -> 60 model inputs
['amt', 'city_pop', 'card_txn_count_1h', 'card_txn_count_24h', 'secs_since_prev_txn', 'amt_to_card_mean_ratio', 'amt_zscore_card', 'card_prior_txn_count', 'missingindicator_secs_since_prev_txn', 'missingindicator_amt_to_card_mean_ratio', 'missingindicator_amt_zscore_card', 'age', 'is_weekend', 'card_category_share', 'missingindicator_card_category_share', 'hour_0', 'hour_1', 'hour_2', 'hour_3', 'hour_4'] ...
tree -> 14 model inputs
['amt', 'city_pop', 'card_txn_count_1h', 'card_txn_count_24h', 'secs_since_prev_txn', 'amt_to_card_mean_ratio', 'amt_zscore_card', 'card_prior_txn_count', 'age', 'is_weekend', 'card_category_share', 'hour', 'day_of_week', 'category'] ...
